# Q4 · C5真实FPGA实验对比
新增多尺度暗线支持和有限连接，保留A/C2。C5是折中实验，亮区误检并未完全解决，原严格质量门槛FAIL保留。
固定20图×两尺寸×A/C2/C5共120帧；逐帧核对真实DMA输出SHA。无HDMI，无PS分割或补线。每框只运行一次，不要Run All。


## 重启后重新开始（2026-10-14）
先通过 http://192.168.2.99:9090/tree 打开Jupyter。在全新目录 retina_q4_c5_restart_r1 中上传本Notebook和同名ZIP，保留旧目录中的未完成记录，不覆盖。
9090是Jupyter文件管理/Notebook；算法实验界面仍为 http://192.168.2.99:8090/，第3框启动服务后才可访问。不要把服务端口改成9090。
如果只关闭了Windows电脑，板上的旧内核可能还在运行。先检查Running和旧Notebook：仅在旧任务正常结束、硬件IDLE且无报错时关闭旧内核；若仍在运行或FAULT，保留内核与输出，先反馈，不加载新Overlay。板卡若也已经重启，旧内核会话已经结束，可从本Notebook第1框开始。
代码框必须从1开始顺序执行；不要按旧页面留下的输出跳过检查或加载。第4框120帧完成后再操作实验网页。


## 1. 验证下载包并展开
应打印 PACKAGE_BYTES_PASS。只读验证通过后写入新的子文件夹；尚未访问 FPGA。

In [ ]:
from pathlib import Path
import hashlib, json, zipfile
if 'browser_package_root' in globals():
    raise RuntimeError('Already extracted; continue to the next cell')
bundle_path = Path('q4_c5_experimental_r1.zip')
expected_bundle_sha256 = 'f49fa30dca7ff0e05135f1497398f255f50adb52021d83b2f92f9687a0775758'
if hashlib.sha256(bundle_path.read_bytes()).hexdigest() != expected_bundle_sha256:
    raise ValueError('Wrong download bytes; stop before importing or loading FPGA')
with zipfile.ZipFile(bundle_path) as archive:
    members = archive.infolist()
    names = [member.filename for member in members]
    manifest = json.loads(archive.read('q4_manifest.json'))
    if len(names) != len(set(names)) or set(names) != set(manifest['artifacts']) | {'q4_manifest.json'}:
        raise ValueError('Unexpected archive inventory')
    if len(members) > 200 or sum(member.file_size for member in members) > 128 * 1024 * 1024:
        raise ValueError('Unexpected package size')
    for member in members:
        path = Path(member.filename)
        if path.is_absolute() or '..' in path.parts or '\\' in member.filename or ':' in member.filename or member.is_dir():
            raise ValueError('Unsafe package member')
        if member.filename != 'q4_manifest.json':
            data = archive.read(member)
            if {'bytes': len(data), 'sha256': hashlib.sha256(data).hexdigest()} != manifest['artifacts'][member.filename]:
                raise ValueError('Package mismatch: ' + member.filename)
    browser_package_root = (Path.cwd() / 'q4_c5_experimental_r1').resolve()
    browser_package_root.mkdir(exist_ok=False)
    for member in members:
        destination = browser_package_root / member.filename
        if browser_package_root not in destination.resolve().parents:
            raise ValueError('Package path escaped directory')
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(member))
print('PACKAGE_BYTES_PASS: paired Q4 and browser application bytes checked; FPGA untouched')


## 2. 检查环境并保留端口
应打印 BROWSER_PRECHECK_PASS 和实际版本。此步不加载FPGA。8090若被占用，保留报错，勿继续。

In [ ]:
import sys
if 'browser_context' in globals():
    raise RuntimeError('Already prepared; continue to the next cell')
sys.path.insert(0, str(browser_package_root))
from q4_board import prepare_q4, start, collect_q4, export_q4, finish
browser_context = prepare_q4(browser_package_root, port=8090)


## 3. 加载一次并启动服务
应打印 BROWSER_SERVICE_READY。尚未进行图像收发。出现红色报错则保留内核和上下文，不重复加载。

In [ ]:
browser_start_status = await start(browser_context)
print(browser_start_status)


## 4. 自动核对120帧
运行后逐帧打印Q4_FRAME_SAVED，最后Q4_COLLECTION_COMPLETE。每帧实际DMA输出SHA须匹配独立参考；全匹配不等于医学质量通过。期间不要在网页提交任务。


In [ ]:
q4_collection = await collect_q4(browser_context)
print(q4_collection)


## 5. 浏览器视觉与参数对照
打开 http://192.168.2.99:8090/ 并Ctrl+F5，页眉须Q4。固定512、最终掩膜模式3、阈值28、快速无损PNG。
按im0001、im0002、im0324顺序，每张先C2运行、再C5运行并截图，共6次。检查中央亮斑附近、主要血管与细血管；C5仅最终模式3启用。
再选im0001/C5，阈值20运行一次、36运行一次；观察参数回读与图像变化。恢复28，点击预热3帧+测量20帧，保持网页前台，结束后截图。
返回运行状态框：预期completed=151、READY/IDLE、pending=0、无error/保留缓冲。多点击产生的真实任务也保留，不要为凑次数重置。


In [ ]:
print(browser_context['manager'].status())


## 6. 导出完整记录
应出现Q4_RESULTS_READY。下载本Notebook同目录q4_results_bundle.zip；不能用普通网页会话导出代替，它不包含固定队列核对记录。


In [ ]:
q4_export_info = await export_q4(browser_context, Path.cwd() / 'q4_results_bundle.zip')


## 7. 安全结束并下载
应出现BROWSER_SERVICE_CLOSED_IDLE。Ctrl+S，再File → Download as → Notebook下载q4_quality_start.ipynb，同时下载q4_results_bundle.zip。
若红色报错或一分钟无新进展：保留输出和内核，不重复执行、不重新加载、不释放失败缓冲。


In [ ]:
await finish(browser_context)
